<a href="https://colab.research.google.com/github/kyeeun7706-coder/e-waste-location-optimization/blob/main/%EC%A0%84%EC%A2%85%EC%84%A40426_%EB%8F%84%EB%B4%89%EA%B5%AC%EC%A7%91%EA%B3%84%EA%B5%AC%EB%B3%84%EA%B2%BD%EC%82%AC%EB%8F%84%EA%B5%AC%ED%95%98%EA%B8%B0(%EB%85%BC%EB%AC%B8%EC%A7%80%ED%91%9C%EC%9D%91%EC%9A%A9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
# [Cell 1] 패키지 설치
# ============================================================
!pip install rasterstats -q


# ============================================================
# [Cell 2] 라이브러리 임포트 + 드라이브 마운트
# ============================================================
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy.ndimage import sobel
from rasterstats import zonal_stats
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# ============================================================
# [Cell 3] 경로 설정
# ============================================================
BASE     = "/content/drive/MyDrive/전종설/0426"
SHP_PATH = f"{BASE}/집계구경계/bnd_oa_11_2025_2Q.shp"
DEM_PATH = f"{BASE}/dem_merged.tif"
OUT_CSV  = f"{BASE}/도봉구_집계구별_평균경사도.csv"
OUT_SHP  = f"{BASE}/도봉구_집계구별_평균경사도.shp"

TARGET_CRS         = "EPSG:5186"
DOBONG_CODE_PREFIX = "11100"   # 도봉구 행정동코드 앞 4자리

In [14]:
# ============================================================
# [Cell 4] 집계구 shapefile 로드 → 도봉구 필터링
# ============================================================
print("▶ 집계구 shapefile 로드 중...")
census = gpd.read_file(SHP_PATH)
print(f"  전체 집계구 수  : {len(census)}")
print(f"  CRS             : {census.crs}")
print(f"  컬럼 목록       : {census.columns.tolist()}")
print(f"\n  첫 행 샘플:\n{census.iloc[0]}\n")

# ADM_CD 기준으로 도봉구 필터링
code_col = "ADM_CD"
dobong = census[census[code_col].astype(str).str.startswith(DOBONG_CODE_PREFIX)].copy()
print(f"  도봉구 집계구 수: {len(dobong)}")

if len(dobong) == 0:
    raise ValueError(f"도봉구 집계구가 0개예요. ADM_CD 샘플값을 확인해주세요: {census[code_col].head().tolist()}")

▶ 집계구 shapefile 로드 중...
  전체 집계구 수  : 19097
  CRS             : EPSG:5179
  컬럼 목록       : ['BASE_DATE', 'ADM_CD', 'TOT_OA_CD', 'geometry']

  첫 행 샘플:
BASE_DATE                                             20250630
ADM_CD                                                11010530
TOT_OA_CD                                       11010530010001
geometry     POLYGON ((953560.7727031708 1953253.582391739,...
Name: 0, dtype: object

  도봉구 집계구 수: 603


In [15]:
# ============================================================
# [Cell 5] DEM 로드 확인
# ============================================================
print("\n▶ DEM(dem_merged.tif) 로드 확인 중...")
with rasterio.open(DEM_PATH) as src:
    print(f"  DEM CRS    : {src.crs}")
    print(f"  DEM 해상도 : {src.res}")
    print(f"  DEM 범위   : {src.bounds}")
    dem_crs = src.crs

# CRS가 5186이 아니면 재투영
if str(dem_crs).upper() != TARGET_CRS:
    print(f"\n  CRS 불일치 → {TARGET_CRS}로 재투영 중...")
    reproj_path = f"{BASE}/dem_merged_5186.tif"
    with rasterio.open(DEM_PATH) as src:
        transform, width, height = calculate_default_transform(
            src.crs, TARGET_CRS, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": TARGET_CRS,
            "transform": transform,
            "width": width,
            "height": height,
            "driver": "GTiff"
        })
        with rasterio.open(reproj_path, "w", **kwargs) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                resampling=Resampling.bilinear
            )
    dem_path_use = reproj_path
    print(f"  재투영 완료 → {reproj_path}")
else:
    dem_path_use = DEM_PATH
    print("  CRS 일치 → 재투영 불필요")


▶ DEM(dem_merged.tif) 로드 확인 중...
  DEM CRS    : EPSG:5179
  DEM 해상도 : (90.0, 90.0)
  DEM 범위   : BoundingBox(left=933390.0, bottom=1916190.0, right=978390.0, top=1972440.0)

  CRS 불일치 → EPSG:5186로 재투영 중...
  재투영 완료 → /content/drive/MyDrive/전종설/0426/dem_merged_5186.tif


In [17]:
# ============================================================
# [Cell 6] DEM → 경사도 래스터 계산
# ============================================================
print("\n▶ 경사도 계산 중...")
with rasterio.open(dem_path_use) as src:
    dem_arr   = src.read(1).astype(float)
    transform = src.transform
    res       = src.res[0]
    nodata    = src.nodata
    profile   = src.profile.copy()

# nodata 마스킹
if nodata is not None:
    nd_mask = (dem_arr == nodata)
    dem_arr[nd_mask] = np.nan

# Sobel 필터 기반 경사도 계산 (단위: 도)
dx = sobel(np.nan_to_num(dem_arr), axis=1) / (8 * res)
dy = sobel(np.nan_to_num(dem_arr), axis=0) / (8 * res)
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

if nodata is not None:
    slope_deg[nd_mask] = -9999

# 경사도 래스터 저장
slope_path = f"{BASE}/slope_merged.tif"
profile.update(driver="GTiff", dtype="float32", nodata=-9999)
with rasterio.open(slope_path, "w", **profile) as dst:
    dst.write(slope_deg.astype("float32"), 1)

print(f"  경사도 저장 완료 → {slope_path}")
# 변경 — nodata(-9999) 제외하고 확인
valid = slope_deg[slope_deg != -9999]
print(f"  경사도 범위: {valid.min():.1f}° ~ {valid.max():.1f}°")


▶ 경사도 계산 중...


  경사도 저장 완료 → /content/drive/MyDrive/전종설/0426/slope_merged.tif
  경사도 범위: 0.0° ~ 75.9°


In [18]:
# ============================================================
# [Cell 7] 집계구 CRS 통일 → Zonal Statistics
# ============================================================
print("\n▶ 집계구별 Zonal Statistics 계산 중...")
dobong = dobong.to_crs(TARGET_CRS)

stats = zonal_stats(
    dobong,
    slope_path,
    stats=["mean", "max", "std", "count"],
    nodata=-9999,
    all_touched=False
)

dobong["경사도_평균"]     = [round(s["mean"], 2) if s["mean"] is not None else None for s in stats]
dobong["경사도_최대"]     = [round(s["max"],  2) if s["max"]  is not None else None for s in stats]
dobong["경사도_표준편차"] = [round(s["std"],  2) if s["std"]  is not None else None for s in stats]
dobong["픽셀수"]          = [s["count"] for s in stats]

print(f"  계산 완료. 유효 집계구: {dobong['경사도_평균'].notna().sum()} / {len(dobong)}")


▶ 집계구별 Zonal Statistics 계산 중...
  계산 완료. 유효 집계구: 530 / 603


In [19]:
# ============================================================
# [Cell 8] 경사도 등급 부여
# ============================================================
def slope_grade(deg):
    if deg is None or np.isnan(deg): return None
    if deg < 3:  return "평지 (α=1.0)"
    if deg < 7:  return "완경사 (α=1.3)"   # 논문 시설한계 7° 적용
    if deg < 9:  return "급경사 (α=1.7)"   # 논문 최대 측정 9° 기준
    return "험준 (α=2.0)"

dobong["경사도_등급"] = dobong["경사도_평균"].apply(slope_grade)

In [26]:
# ============================================================
# [Cell 9] 결과 저장
# ============================================================
OUT_CSV_논문 = f"{BASE}/도봉구_집계구별_평균경사도(논문지표응용).csv"
OUT_SHP_논문 = f"{BASE}/도봉구_집계구별_평균경사도(논문지표응용).shp"

print("\n▶ 결과 저장 중...")

df_save = dobong.drop(columns="geometry").copy()
df_save["ADM_CD"]    = df_save["ADM_CD"].astype(str)
df_save["TOT_OA_CD"] = df_save["TOT_OA_CD"].astype(str)
df_save.to_csv(OUT_CSV_논문, index=False, encoding="utf-8-sig")

dobong.to_file(OUT_SHP_논문, encoding="utf-8")

print(f"  CSV 저장 완료 → {OUT_CSV_논문}")
print(f"  SHP 저장 완료 → {OUT_SHP_논문}")


# ============================================================
# [Cell 10] 요약 출력
# ============================================================
print("\n" + "="*55)
print("  도봉구 집계구별 평균 경사도 결과 요약 (논문지표응용)")
print("="*55)
print(f"  전체 집계구 수          : {len(dobong)}")
print(f"  전체 평균 경사도        : {dobong['경사도_평균'].mean():.2f}°")
print(f"  DEM 미커버 집계구       : {(dobong['픽셀수'] == 0).sum()}개")
print(f"\n  등급별 집계구 수 (논문 기준):")
print(dobong["경사도_등급"].value_counts().to_string())
print(f"\n  경사도 상위 10개 집계구:")
print(
    dobong.nlargest(10, "경사도_평균")[["ADM_CD", "TOT_OA_CD", "경사도_평균", "경사도_등급"]]
    .to_string(index=False))


▶ 결과 저장 중...
  CSV 저장 완료 → /content/drive/MyDrive/전종설/0426/도봉구_집계구별_평균경사도(논문지표응용).csv
  SHP 저장 완료 → /content/drive/MyDrive/전종설/0426/도봉구_집계구별_평균경사도(논문지표응용).shp

  도봉구 집계구별 평균 경사도 결과 요약 (논문지표응용)
  전체 집계구 수          : 603
  전체 평균 경사도        : 1.76°
  DEM 미커버 집계구       : 73개

  등급별 집계구 수 (논문 기준):
경사도_등급
평지 (α=1.0)     438
완경사 (α=1.3)     87
급경사 (α=1.7)      4
험준 (α=2.0)       1

  경사도 상위 10개 집계구:
  ADM_CD      TOT_OA_CD  경사도_평균      경사도_등급
11100640 11100640010004   15.35  험준 (α=2.0)
11100640 11100640010009   11.76 급경사 (α=1.7)
11100560 11100560010012    9.82 급경사 (α=1.7)
11100610 11100610010001    8.28 급경사 (α=1.7)
11100640 11100640010005    8.08 급경사 (α=1.7)
11100570 11100570010001    7.75 완경사 (α=1.3)
11100510 11100510020027    7.26 완경사 (α=1.3)
11100510 11100510020029    6.86 완경사 (α=1.3)
11100510 11100510020017    6.74 완경사 (α=1.3)
11100510 11100510020015    6.55 완경사 (α=1.3)


/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: '경사도_평균' to '경사도_'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Could not decode error message to UTF-8. Raw error: b"Normalized/laundered field name: '\xea\xb2\xbd\xec\x82\xac\xeb\x8f\x84_\xec\xb5\x9c\xeb\x8c\x80' to '\xea\xb2\xbd\xec\x82\xac\xeb\x8f_1'"
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Could not decode error message to UTF-8. Raw error: b"Normalized/laundered field name: '\xea\xb2\xbd\xec\x82\xac\xeb\x8f\x84_\xed\x91\x9c\xec\xa4\x80\xed\x8e\xb8\xec\xb0\xa8' to '\xea\xb2\xbd\xec\x82\xac\xeb\x8f_2'"
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Could not decode error message to UTF-8. Raw error: b"Normalized/laundered field name: '\xea\xb2\xbd\xec\x82\xac\xeb\x8f\x84_\xeb\x93\xb1\xea\xb8\x89' to '\xea\xb2\xbd\xec\x82\xac\xeb\x8f_3'"
  ogr